In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS, load_scaler, score_split

SEED = 11
torch.manual_seed(SEED); np.random.seed(SEED)
cols, scaler = load_scaler()
print(len(cols), "features")

40 features


In [2]:
def benign_matrix(name, n):
    df = pd.read_parquet(DATA / f"{name}.parquet", columns=cols)
    df = df.sample(min(n, len(df)), random_state=SEED)
    return scaler.transform(to_matrix(df, cols)).astype(np.float32)

x_train = benign_matrix("train", 1_000_000)
x_val = benign_matrix("val", 100_000)
print(x_train.shape, x_val.shape)

(1000000, 40) (100000, 40)


In [3]:
d = x_train.shape[1]
model = nn.Sequential(
    nn.Linear(d, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 12), nn.ReLU(),     # bottleneck
    nn.Linear(12, 32), nn.ReLU(),
    nn.Linear(32, 64), nn.ReLU(),
    nn.Linear(64, d),
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
Xtr, Xva = torch.from_numpy(x_train), torch.from_numpy(x_val)

best, best_state, bad = float("inf"), None, 0
for epoch in range(30):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 1024):
        xb = Xtr[perm[i:i + 1024]]
        opt.zero_grad()
        loss_fn(model(xb), xb).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        v = loss_fn(model(Xva), Xva).item()
    print(f"epoch {epoch + 1:2d}  val loss {v:.4f}")
    if v < best - 1e-4:
        best, bad = v, 0
        best_state = {k: t.clone() for k, t in model.state_dict().items()}
    else:
        bad += 1
        if bad >= 3:
            break
model.load_state_dict(best_state)

epoch  1  val loss 0.2949
epoch  2  val loss 0.1864
epoch  3  val loss 0.2271
epoch  4  val loss 0.1315
epoch  5  val loss 0.1885
epoch  6  val loss 0.0988
epoch  7  val loss 0.0740
epoch  8  val loss 0.0803
epoch  9  val loss 0.0895
epoch 10  val loss 0.1056


<All keys matched successfully>

In [4]:
def ae_score(x):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(x), 100_000):
            xb = torch.from_numpy(x[i:i + 100_000])
            out.append(((model(xb) - xb) ** 2).mean(dim=1).numpy())
    return np.concatenate(out)

val_s = score_split("val", cols, scaler, ae_score)
tune_s = score_split("tune", cols, scaler, ae_score)
test_s = score_split("test", cols, scaler, ae_score, cap=100_000)

for name, d_ in [("tune", tune_s), ("test", test_s)]:
    r = evaluate(d_, val_df=val_s)
    print(name, {k: round(v, 3) for k, v in r.items() if k != "per_attack_tpr@fpr0.01"})
    print("  per-attack TPR@1%FPR:", {k: round(v, 3) for k, v in r["per_attack_tpr@fpr0.01"].items()})

(RESULTS / "scores").mkdir(parents=True, exist_ok=True)
pd.concat([val_s, tune_s, test_s]).to_parquet(
    RESULTS / "scores" / f"m3_autoencoder_nftoniot_seed{SEED}.parquet")

tune {'auroc': 0.877, 'aupr': 0.694, 'tpr@fpr0.001': 0.001, 'real_fpr@0.001': 0.002, 'tpr@fpr0.01': 0.018, 'real_fpr@0.01': 0.013, 'tpr@fpr0.05': 0.07, 'real_fpr@0.05': 0.061}
  per-attack TPR@1%FPR: {'ddos': 0.019, 'dos': 0.008, 'injection': 0.311, 'mitm': 0.0, 'password': 0.026, 'scanning': 0.014, 'xss': 0.031}
test {'auroc': 0.433, 'aupr': 0.32, 'tpr@fpr0.001': 0.109, 'real_fpr@0.001': 0.116, 'tpr@fpr0.01': 0.236, 'real_fpr@0.01': 0.269, 'tpr@fpr0.05': 0.812, 'real_fpr@0.05': 0.808}
  per-attack TPR@1%FPR: {'backdoor': 0.997, 'ddos': 0.186, 'dos': 0.151, 'injection': 0.267, 'mitm': 0.274, 'password': 0.443, 'ransomware': 0.792, 'scanning': 0.108, 'xss': 0.114}


In [5]:
# 1. Oracle thresholds: benign scores of the same split set the cutoff
for name, d_ in [("tune", tune_s), ("test", test_s)]:
    r = evaluate(d_)
    print(name, "ORACLE", {k: round(r[k], 3) for k in ["auroc", "tpr@fpr0.001", "tpr@fpr0.01", "tpr@fpr0.05"]})

# 2. Same oracle view for M1, using its saved score file (same test sample, same seed)
m1 = pd.read_parquet(RESULTS / "scores" / "m1_iforest_nftoniot_seed11.parquet")
r1 = evaluate(m1[m1["split"] == "test"])
print("M1 test ORACLE", {k: round(r1[k], 3) for k in ["auroc", "tpr@fpr0.001", "tpr@fpr0.01", "tpr@fpr0.05"]})

# 3. Score quantiles: where do benign tails and attacks sit?
qs = [0.5, 0.9, 0.99, 0.999]
print(pd.DataFrame({
    "val_benign": val_s["score"].quantile(qs),
    "tune_benign": tune_s[tune_s["label"] == 0]["score"].quantile(qs),
    "test_benign": test_s[test_s["label"] == 0]["score"].quantile(qs)}))
print(tune_s[tune_s["label"] == 1].groupby("attack_type")["score"].median())

tune ORACLE {'auroc': 0.877, 'tpr@fpr0.001': 0.001, 'tpr@fpr0.01': 0.017, 'tpr@fpr0.05': 0.061}
test ORACLE {'auroc': 0.433, 'tpr@fpr0.001': 0.0, 'tpr@fpr0.01': 0.004, 'tpr@fpr0.05': 0.031}
M1 test ORACLE {'auroc': 0.52, 'tpr@fpr0.001': 0.0, 'tpr@fpr0.01': 0.002, 'tpr@fpr0.05': 0.089}
       val_benign  tune_benign  test_benign
0.500    0.000264     0.000264     0.603272
0.900    0.014625     0.091785    11.076944
0.990    0.804277     0.970626    43.052567
0.999    3.569258     6.883743    98.920100
attack_type
ddos         0.233813
dos          0.286800
injection    0.353555
mitm         0.142915
password     0.223739
scanning     0.061124
xss          0.284094
Name: score, dtype: float32


In [6]:
from src.pipeline import iter_split

def get(name, cap=30_000, n_benign=100_000):
    df = pd.concat(list(iter_split(name, cols, cap=cap)))
    b = df[df["Attack"] == "Benign"]
    b = b.sample(min(n_benign, len(b)), random_state=SEED)
    return pd.concat([b, df[df["Attack"] != "Benign"]])

tr = pd.read_parquet(DATA / "train.parquet", columns=cols + ["Attack"]).sample(200_000, random_state=SEED)
va, tu, te = get("val"), get("tune"), get("test")

groups = {
    "train_benign": tr,
    "val_benign": va,
    "tune_benign": tu[tu["Attack"] == "Benign"],
    "tune_attack": tu[tu["Attack"] != "Benign"],
    "test_benign": te[te["Attack"] == "Benign"],
    "test_attack": te[te["Attack"] != "Benign"],
}
feats = [c for c in ["DURATION_OUT", "DURATION_IN", "RETRANSMITTED_IN_PKTS",
                     "RETRANSMITTED_IN_BYTES", "IN_BYTES", "LONGEST_FLOW_PKT"] if c in cols]

print("fraction of flows with value > 0")
print(pd.DataFrame({g: (d[feats] > 0).mean() for g, d in groups.items()}).round(3))
print("\n99th percentile (raw values)")
print(pd.DataFrame({g: d[feats].quantile(0.99) for g, d in groups.items()}).round(1))
print("\ntrain std after log1p (what the scaler divides by)")
print(pd.Series(scaler.scale_, index=cols)[feats].round(4))

fraction of flows with value > 0
                        train_benign  val_benign  tune_benign  tune_attack  \
DURATION_OUT                   0.002       0.003        0.004        0.010   
DURATION_IN                    0.008       0.017        0.011        0.018   
RETRANSMITTED_IN_PKTS          0.000       0.000        0.001        0.001   
RETRANSMITTED_IN_BYTES         0.000       0.000        0.001        0.001   
IN_BYTES                       1.000       1.000        1.000        1.000   
LONGEST_FLOW_PKT               1.000       1.000        1.000        1.000   

                        test_benign  test_attack  
DURATION_OUT                  0.265        0.223  
DURATION_IN                   0.334        0.223  
RETRANSMITTED_IN_PKTS         0.047        0.026  
RETRANSMITTED_IN_BYTES        0.047        0.026  
IN_BYTES                      1.000        1.000  
LONGEST_FLOW_PKT              1.000        1.000  

99th percentile (raw values)
                        train_ben